<a href="https://colab.research.google.com/github/Sreya889/Semeval-Task-13-Subtask-A/blob/main/Codebert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re, random, time
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import f1_score, confusion_matrix, classification_report


# PATHS & CONFIG
BASE_PATH = "/content/drive/MyDrive/SemEval-Task13"
DATA_PATH = f"{BASE_PATH}/cleaned_data"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/codebert_antioverfit"
FINAL_MODEL_PATH = f"{BASE_PATH}/models/codebert_final_antioverfit"

os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)

# TO REDUCE OVERFITTING
RANDOM_SEED = 42
TRAIN_SIZE = 100_000      # REDUCED from 120k to prevent memorization
VAL_SIZE = 20_000
MAX_LENGTH = 256
MODEL_NAME = "microsoft/codebert-base"
BATCH_SIZE = 16           # REDUCED from 20 for more gradient updates
GRAD_ACCUM = 3            # Effective batch = 48
EPOCHS = 3                # INCREASED from 2 for better learning

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


start_time = time.time()


print("\n Loading data...")
train_df = pd.read_parquet(f"{DATA_PATH}/train_cleaned.parquet")
val_df   = pd.read_parquet(f"{DATA_PATH}/val_cleaned.parquet")
full_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Total samples: {len(full_df):,}")
print(f"Label distribution: {full_df['label'].value_counts().to_dict()}")

# GENERATOR FAMILIES

generator_families = {
    "qwen": ['Qwen/Qwen2.5-Coder-1.5B','Qwen/Qwen2.5-Coder-7B',
             'Qwen/Qwen2.5-Coder-7B-Instruct','Qwen/Qwen2.5-Coder-32B-Instruct',
             'Qwen/Qwen2.5-Coder-1.5B-Instruct'],
    "phi": ['microsoft/phi-2','microsoft/Phi-3-mini-4k-instruct',
            'microsoft/Phi-3-medium-4k-instruct','microsoft/Phi-3-small-8k-instruct',
            'microsoft/Phi-3.5-mini-instruct'],
    "codellama": ['codellama/CodeLlama-7b-hf',
                  'codellama/CodeLlama-34b-Instruct-hf',
                  'codellama/CodeLlama-70b-Instruct-hf'],
    "yi": ['01-ai/Yi-Coder-1.5B','01-ai/Yi-Coder-9B',
           '01-ai/Yi-Coder-1.5B-Chat','01-ai/Yi-Coder-9B-Chat'],
    "llama": ['meta-llama/Llama-3.1-8B','meta-llama/Llama-3.1-8B-Instruct',
              'meta-llama/Llama-3.2-1B','meta-llama/Llama-3.2-3B',
              'meta-llama/Llama-3.3-70B-Instruct'],
    "deepseek": ['deepseek-ai/deepseek-coder-1.3b-base',
                 'deepseek-ai/deepseek-coder-1.3b-instruct',
                 'deepseek-ai/deepseek-coder-6.7b-base',
                 'deepseek-ai/deepseek-coder-6.7b-instruct'],
    "starcoder": ['bigcode/starcoder','bigcode/starcoder2-3b',
                  'bigcode/starcoder2-7b','bigcode/starcoder2-15b'],
    "granite": ['ibm-granite/granite-8b-code-base-4k',
                'ibm-granite/granite-8b-code-instruct-4k'],
    "codegemma": ['google/codegemma-2b','google/codegemma-7b']
}

VAL_FAMILIES = ["llama", "deepseek", "yi"]
val_ai_generators = [g for fam in VAL_FAMILIES for g in generator_families[fam]]

# TRAIN/VAL SPLIT
print("\nCreating train/val split...")

human_data = full_df[full_df['generator'] == 'human'].copy()
ai_data = full_df[full_df['generator'] != 'human'].copy()

val_ai_data = ai_data[ai_data["generator"].isin(val_ai_generators)].copy()
train_ai_data = ai_data[~ai_data["generator"].isin(val_ai_generators)].copy()

human_shuffled = human_data.sample(frac=1, random_state=RANDOM_SEED)
human_train_size = len(human_shuffled) // 2
train_human_data = human_shuffled.iloc[:human_train_size].copy()
val_human_data = human_shuffled.iloc[human_train_size:].copy()

train_df_full = pd.concat([train_ai_data, train_human_data], ignore_index=True)
val_df_full = pd.concat([val_ai_data, val_human_data], ignore_index=True)

# BALANCED SAMPLING
print("\nCreating balanced datasets...")

# Training: 50k human, 50k AI
target_human = TRAIN_SIZE // 2
target_ai = TRAIN_SIZE // 2

train_human_sampled = train_human_data.sample(
    n=min(target_human, len(train_human_data)),
    random_state=RANDOM_SEED
)

train_ai_generators = train_ai_data['generator'].unique()
ai_per_generator = target_ai // len(train_ai_generators)

ai_samples = []
for gen in train_ai_generators:
    gen_data = train_ai_data[train_ai_data['generator'] == gen]
    sample_size = min(len(gen_data), ai_per_generator)
    ai_samples.append(gen_data.sample(n=sample_size, random_state=RANDOM_SEED))

train_ai_sampled = pd.concat(ai_samples, ignore_index=True)

if len(train_ai_sampled) > target_ai:
    train_ai_sampled = train_ai_sampled.sample(n=target_ai, random_state=RANDOM_SEED)
elif len(train_ai_sampled) < target_ai:
    remaining = target_ai - len(train_ai_sampled)
    extra = train_ai_data[~train_ai_data.index.isin(train_ai_sampled.index)].sample(
        n=min(remaining, len(train_ai_data) - len(train_ai_sampled)),
        random_state=RANDOM_SEED
    )
    train_ai_sampled = pd.concat([train_ai_sampled, extra], ignore_index=True)

train_df = pd.concat([train_human_sampled, train_ai_sampled], ignore_index=True)
train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Validation: 10k human, 10k AI
target_val_human = VAL_SIZE // 2
target_val_ai = VAL_SIZE // 2

val_human_sampled = val_human_data.sample(
    n=min(target_val_human, len(val_human_data)),
    random_state=RANDOM_SEED
)

val_ai_sampled = val_ai_data.sample(
    n=min(target_val_ai, len(val_ai_data)),
    random_state=RANDOM_SEED
)

val_df = pd.concat([val_human_sampled, val_ai_sampled], ignore_index=True)
val_df = val_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# VERIFY DISTRIBUTIONS
print("\n" + "="*70)
print("DATASET DISTRIBUTIONS")
print("="*70)

print("\nTRAINING:")
print(f"  Total: {len(train_df):,}")
train_label_dist = train_df['label'].value_counts()
print(f"  Human: {train_label_dist.get(0, 0):,} ({train_label_dist.get(0, 0)/len(train_df)*100:.1f}%)")
print(f"  AI: {train_label_dist.get(1, 0):,} ({train_label_dist.get(1, 0)/len(train_df)*100:.1f}%)")

print("\nVALIDATION:")
print(f"  Total: {len(val_df):,}")
val_label_dist = val_df['label'].value_counts()
print(f"  Human: {val_label_dist.get(0, 0):,} ({val_label_dist.get(0, 0)/len(val_df)*100:.1f}%)")
print(f"  AI: {val_label_dist.get(1, 0):,} ({val_label_dist.get(1, 0)/len(val_df)*100:.1f}%)")
print(f"  OOD families: {VAL_FAMILIES}")

assert val_label_dist.get(0, 0) > 0, "NO HUMAN SAMPLES IN VALIDATION!"
assert val_label_dist.get(1, 0) > 0, "NO AI SAMPLES IN VALIDATION!"

print("\nSanity checks passed")
print("="*70)

# AGGRESSIVE AUGMENTATION TO PREVENT MEMORIZATION
def aggressive_augment(examples):
    augmented_codes = []

    for code in examples["code"]:
        # 80% augmentation rate
        if random.random() < 0.8:
            # Variable renaming (50% of augmented samples)
            if random.random() < 0.5:
                vars_ = set(re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code))

                keywords = {
                    'def','class','if','else','elif','for','while','return',
                    'import','from','print','try','except','finally','with',
                    'as','pass','break','continue','raise','yield','lambda',
                    'True','False','None','and','or','not','in','is','self',
                    'len','range','str','int','float','list','dict','set',
                    'open','close','read','write','append','extend','update',
                    'get','put','pop','push','insert','remove','delete','clear'
                }

                maskable = sorted(vars_ - keywords)
                var_map = {v: f'var_{i}' for i, v in enumerate(maskable)}

                for orig, masked in var_map.items():
                    code = re.sub(r'\b' + re.escape(orig) + r'\b', masked, code)

            # Whitespace/formatting changes (30% of augmented samples)
            elif random.random() < 0.6:
                # Remove extra spaces
                code = re.sub(r' +', ' ', code)
                # Normalize line breaks
                code = re.sub(r'\n\n+', '\n', code)

            # Comment removal (20% of augmented samples)
            else:
                # Remove single-line comments
                code = re.sub(r'#.*$', '', code, flags=re.MULTILINE)
                # Remove docstrings
                code = re.sub(r'""".*?"""', '', code, flags=re.DOTALL)
                code = re.sub(r"'''.*?'''", '', code, flags=re.DOTALL)

        augmented_codes.append(code)

    examples["code"] = augmented_codes
    return examples

# TOKENIZATION
print("\nTokenizing...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["code", "label"]])
val_ds   = Dataset.from_pandas(val_df[["code", "label"]])

print("Applying AGGRESSIVE augmentation (80% rate)...")
train_ds = train_ds.map(aggressive_augment, batched=True, batch_size=1000)

def tokenize(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["code"])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=["code"])

train_ds = train_ds.rename_column("label", "labels").with_format("torch")
val_ds   = val_ds.rename_column("label", "labels").with_format("torch")

# MODEL WITH HEAVY REGULARIZATION
print("\nLoading model with heavy regularization...")
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=0.3,           # INCREASED from 0.2
    attention_probs_dropout_prob=0.3   # INCREASED from 0.2
)
model.gradient_checkpointing_enable()

# METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    cm = confusion_matrix(labels, preds)
    f1_per_class = f1_score(labels, preds, average=None)

    metrics = {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": (preds == labels).mean(),
        "f1_human": f1_per_class[0] if len(f1_per_class) > 0 else 0.0,
        "f1_ai": f1_per_class[1] if len(f1_per_class) > 1 else 0.0,
    }

    print("\n" + "="*50)
    print("CONFUSION MATRIX:")
    print("              Predicted")
    print("            Human    AI")
    print(f"True Human: {cm[0,0]:5d}  {cm[0,1]:5d}")
    print(f"True AI:    {cm[1,0]:5d}  {cm[1,1]:5d}")
    print(f"\nF1 Human: {metrics['f1_human']:.4f}")
    print(f"F1 AI: {metrics['f1_ai']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")

    return metrics

# TRAINING ARGS
training_args = TrainingArguments(
    output_dir=CHECKPOINT_PATH,
    eval_strategy="steps",
    save_strategy="steps",

    learning_rate=1.5e-5,              # REDUCED from 2e-5 for stability
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,                  # CHANGED to ratio for stability
    weight_decay=0.02,                 # INCREASED from 0.01
    label_smoothing_factor=0.15,       # INCREASED from 0.1

    eval_steps=1000,                   # MORE FREQUENT evals
    save_steps=1000,
    logging_steps=250,

    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,
    dataloader_num_workers=2,
    group_by_length=False,             # DISABLED - can cause overfitting
    report_to="none",

    max_grad_norm=0.5,                 # REDUCED from 1.0 for stability
    optim="adamw_torch"
)

# TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # INCREASED patience
)

# TRAIN
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Total steps: ~{len(train_ds) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}")

last_ckpt = get_last_checkpoint(CHECKPOINT_PATH)
trainer.train(resume_from_checkpoint=last_ckpt)

train_time = (time.time() - start_time) / 3600

# FINAL EVALUATION
print("\nFinal evaluation")
final_metrics = trainer.evaluate()

predictions = trainer.predict(val_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(labels, preds, target_names=['Human', 'AI'], digits=4))

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Final Macro F1: {final_metrics['eval_macro_f1']:.4f}")
print(f"Final F1 Human: {final_metrics['eval_f1_human']:.4f}")
print(f"Final F1 AI: {final_metrics['eval_f1_ai']:.4f}")
print(f"Final Accuracy: {final_metrics['eval_accuracy']:.4f}")

# SAVE MODEL
print(f"\nSaving model...")
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

print("\n SUMMARY:")
print(f"   • Training: {len(train_df):,} samples (50/50 human/AI)")
print(f"   • Validation: {len(val_df):,} samples (50/50 human/AI, OOD)")
print(f"   • Augmentation: 80% rate")
print(f"   • Dropout: 0.3 (heavy)")
print(f"   • Time: {train_time:.2f} hours")
print(f"   • Final macro F1: {final_metrics['eval_macro_f1']:.4f}")
print(f"\n Model saved: {FINAL_MODEL_PATH}")

In [ ]:
from datasets import Dataset
from transformers import RobertaTokenizer
import pandas as pd

TEST_PATH = "/content/drive/MyDrive/SemEval-Task13/test_data/test.parquet"
TOKEN_SAVE_PATH = "/content/drive/MyDrive/SemEval-Task13/tokenized_test"

print("Loading test data...")
test_df = pd.read_parquet(TEST_PATH)
print(f"Loaded {len(test_df):,} samples")

tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

dataset = Dataset.from_pandas(
    test_df[["code"]],
    preserve_index=False
)

def tokenize(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

print("🔧 Tokenizing with progress bar...")
dataset = dataset.map(
    tokenize,
    batched=True,
    num_proc=1,
    remove_columns=["code"],
    desc="Tokenizing"
)

dataset.set_format(type="torch")

print("Saving tokenized dataset to Drive (IMPORTANT)...")
dataset.save_to_disk(TOKEN_SAVE_PATH)

print("Tokenization complete and SAVED")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import zipfile
import shutil
from google.colab import drive

# 1. MOUNT AND SET PATHS
drive.mount('/content/drive', force_remount=True)

MODEL_PATH = "/content/drive/MyDrive/SemEval-Task13/checkpoints/codebert_antioverfit/checkpoint-1000"
TOKENIZED_TEST_PATH = "/content/drive/MyDrive/SemEval-Task13/tokenized_test"
TEST_DATA_PATH = "/content/drive/MyDrive/SemEval-Task13/test_data/test.parquet"
DRIVE_BACKUP_FOLDER = "/content/drive/MyDrive/SemEval-Task13/"

print("="*70)
print("CodeBERT Checkpoint-1000 → Submission (Pre-Tokenized)")
print("="*70)

# 2. LOAD MODEL
print("\n Loading CodeBERT from checkpoint-1000...")
model = RobertaForSequenceClassification.from_pretrained(MODEL_PATH)
print("Model loaded")

# 3. LOAD PRE-TOKENIZED DATA
print("\n Loading pre-tokenized test data...")
tokenized_test = load_from_disk(TOKENIZED_TEST_PATH)
print(f"Loaded {len(tokenized_test):,} tokenized samples (no re-tokenization needed!)")

# 4. LOAD IDS
print("\nLoading test IDs...")
test_df = pd.read_parquet(TEST_DATA_PATH)
id_column = "id" if "id" in test_df.columns else "ID"
test_ids = test_df[id_column].tolist()
print(f"Loaded {len(test_ids):,} IDs")

# 5. FAST INFERENCE WITH TRAINER
print("\n Running inference (15-20 mins expected)...")

test_args = TrainingArguments(
    output_dir="./temp",
    per_device_eval_batch_size=64,
    fp16=True,
    eval_accumulation_steps=50,
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(model=model, args=test_args)
raw_predictions = trainer.predict(tokenized_test)
predicted_labels = np.argmax(raw_predictions.predictions, axis=-1)

print(f"Generated {len(predicted_labels):,} predictions!")

# 6. CREATE SUBMISSION
print("\nCreating submission file...")

submission_df = pd.DataFrame({
    "id": test_ids,
    "label": predicted_labels.astype(int)
})

# 7. SAVE AND BACKUP
csv_name = "submission_codebert_ckpt1000.csv"
zip_name = "submission_codebert_ckpt1000.zip"

submission_df.to_csv(csv_name, index=False)

with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(csv_name)

# Backup to Drive
shutil.copy(zip_name, os.path.join(DRIVE_BACKUP_FOLDER, zip_name))

print(f"\nFINISHED! Total predictions: {len(submission_df):,}")

# 8. STATISTICS
print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

n_human = (submission_df["label"] == 0).sum()
n_ai = (submission_df["label"] == 1).sum()

print(f"Human (0): {n_human:,} ({n_human/len(submission_df)*100:.1f}%)")
print(f"AI (1): {n_ai:,} ({n_ai/len(submission_df)*100:.1f}%)")

print("\nFirst 10 predictions:")
print(submission_df.head(10))